### Investigation on the all-or-none model
Note that the expected degree, which is $\frac{1}{2} \times E$, satisfies the following recurrence relation:
$$d_N = d_{N-1} \frac{(N-1+2p)}{N} + \frac{2}{N}$$
With initial conditions being $d_1 = 0, d_2 = 1$

In [ ]:
def d(N, p):
    if N == 1:
        return 0.0
    if N == 2:
        return 1.0  # E=1, d=2*1/2=1
    d_prev = 1.0
    for n in range(3, N+1):
        d_prev = d_prev * ( (n-1) + 2*p ) / n + 2 / n
    return d_prev

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from random import choice

class AllOrNoneDynamics:
    def __init__(self, p, initial_nodes=1):
        self.graph = nx.Graph()
        self.graph.add_nodes_from(range(initial_nodes))
        self.current_node_id = initial_nodes  
        self.threshold = p 
        self.iteration_log = []  
    
    def select_target_node(self):
        existing_nodes = list(self.graph.nodes)
        if not existing_nodes:
            raise ValueError("No existing nodes in the network!")
        return choice(existing_nodes)
    
    def add_new_node(self):
        target_node = self.select_target_node()
        new_node = self.current_node_id
        self.graph.add_node(new_node)
        self.current_node_id += 1
        
        p = self.threshold
        judge_value = np.random.uniform(0, 1)
        
        if judge_value < p:
            # Clone (Probability p)
            connect_nodes = [target_node] + list(self.graph.neighbors(target_node))
            action = f"Node {new_node} Cloned target {target_node}"
        else:
            # Leaf (Probability 1-p)
            connect_nodes = [target_node]
            action = f"Node {new_node} added as Leaf to target {target_node}"
        
        connect_nodes = list(set(connect_nodes))  
        for node in connect_nodes:
            self.graph.add_edge(new_node, node)
            
        self.iteration_log.append(action)
        return action
    
    def run_dynamics(self, total_nodes=20):
        if total_nodes <= len(self.graph.nodes):
            return self.iteration_log
        
        for i in range(total_nodes - len(self.graph.nodes)):
            log = self.add_new_node()
            
        print(f"\nEvolution completed! Final network: {len(self.graph.nodes)} nodes, {len(self.graph.edges)} edges")
        return self.iteration_log
    
    def visualize_network(self):
        """Enhanced visualization showing hubs, scale, and structure."""
        plt.figure(figsize=(12, 12), facecolor='#f8f9fa')
        
        # Calculate degrees for sizing and coloring
        degrees = dict(self.graph.degree())
        node_colors = [v for v in degrees.values()] # Use degree for color intensity
        
        # Kamada-Kawai often produces cleaner force-directed layouts than spring_layout
        pos = nx.kamada_kawai_layout(self.graph)  
        
        nx.draw_networkx_edges(
            self.graph, pos, 
            edge_color="#A0AEC0", 
            width=0.6, 
            alpha=0.4
        )
        
        # Draw nodes with a nice colormap (plasma or viridis work well)
        nodes = nx.draw_networkx_nodes(
            self.graph, pos,  
            node_color=node_colors, 
            cmap=plt.cm.plasma, 
            edgecolors="white", 
            linewidths=1.5,
            alpha=0.95
        )
        
        # Add a subtle colorbar to show what colors mean
        cbar = plt.colorbar(nodes, shrink=0.5, pad=0.02, aspect=30)
        cbar.set_label('Node Degree', rotation=270, labelpad=15, fontsize=12)
        
        plt.title(f"All-or-None Model (N={len(self.graph.nodes)}, p={self.threshold})", fontsize=16, fontweight='bold', pad=20)
        plt.axis("off")
        plt.tight_layout
        plt.show()

if __name__ == "__main__":
    model = AllOrNoneDynamics(initial_nodes=1, p=0.7)
    log = model.run_dynamics(total_nodes=200)
    model.visualize_network()
    

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Corrected Model Class Definition
# ==========================================
class AllOrNoneDynamics:
    """
    All-or-None Dynamics Model tracking Total Number of Links
    """
    def __init__(self, p):
        self.graph = nx.Graph()
        # Initialize with 2 nodes and 1 edge (N=2, L=1)
        self.graph.add_edge(0, 1)
        self.current_node_id = 2  
        self.p = p 
        
        # CRITICAL FIX: Track total edges as NUMBERS, starting at N=2
        self.edge_history = [1] 
    
    def add_new_node(self):
        target = np.random.randint(0, self.current_node_id)
        new_node = self.current_node_id
        self.graph.add_node(new_node)
        self.current_node_id += 1
        
        judge_value = np.random.uniform(0, 1)
        
        if judge_value < self.p:
            # Clone (Probability p)
            connect_nodes = [target] + list(self.graph.neighbors(target))
        else:
            # Leaf (Probability 1-p)
            connect_nodes = [target]
        
        # Establish connections
        connect_nodes = list(set(connect_nodes))  
        for node in connect_nodes:
            self.graph.add_edge(new_node, node)
            
        # Append the integer count of edges, NOT text
        self.edge_history.append(self.graph.number_of_edges())
    
    def run_dynamics(self, total_nodes):
        """Runs the growth and returns the edge history numerical array."""
        while self.current_node_id < total_nodes:
            self.add_new_node()
        # Return the numerical array so np.mean() can process it
        return np.array(self.edge_history)

# ==========================================
# 2. Theoretical Difference Equation
# ==========================================
def compute_theoretical_links(p, N_max):
    """
    Computes the theoretical L_N using the derived recurrence:
    L_N = L_{N-1} * (1 + 2p / (N-1)) + 1
    With base conditions L_1 = 0, L_2 = 1
    """
    L_vals = np.zeros(N_max + 1)
    
    # Base cases
    L_vals[1] = 0.0
    L_vals[2] = 1.0
    
    # Calculate step-by-step up to N_max
    for N in range(3, N_max + 1):
        L_vals[N] = L_vals[N-1] * (1 + 2 * p / (N - 1)) + 1
        
    # Return from N=2 upwards to match the empirical data tracking
    return L_vals[2:]  

# ==========================================
# 3. Simulation & Plotting
# ==========================================
N_max = 1000
n_iter = 30  # Independent runs to smooth the empirical data
p_values = [0.1, 0.3, 0.5, 0.7]

sns.set_style("whitegrid", {"grid.linestyle": "--", "grid.alpha": 0.5})
plt.figure(figsize=(12, 8))

# Define a distinct color palette
colors = sns.color_palette("Set1", len(p_values))
N_range = np.arange(2, N_max + 1)

for idx, p in enumerate(p_values):
    print(f"Simulating p = {p}...")
    
    # 1. Gather Empirical Data (Total Links)
    empirical_runs_edges = []
    for _ in range(n_iter):
        model = AllOrNoneDynamics(p=p)
        history = model.run_dynamics(total_nodes=N_max)
        empirical_runs_edges.append(history)
    
    # Average total edges across all iterations. 
    # This will now work perfectly because 'history' contains numbers!
    empirical_avg_edges = np.mean(empirical_runs_edges, axis=0)
    
    # 2. Gather Theoretical Data (Total Links)
    theoretical_L = compute_theoretical_links(p, N_max)
    
    # 3. Plotting
    # Empirical (Solid line)
    plt.plot(N_range, empirical_avg_edges, label=f'Empirical (p={p})', 
             color=colors[idx], linewidth=3, alpha=0.6)
    
    # Theoretical (Dashed line)
    plt.plot(N_range, theoretical_L, 
             color=colors[idx], linestyle='--', linewidth=2, 
             dash_capstyle='round', dashes=[4, 3])

# Fake lines for legend formatting
plt.plot([], [], color='gray', linewidth=3, alpha=0.6, label='Empirical L(N)')
plt.plot([], [], color='gray', linestyle='--', linewidth=2, label='Theoretical L(N)')

# Formatting
plt.title(f'Total Number of ln(Links) vs. ln(Network Size)\n(Averaged over {n_iter} runs)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Log of Network Size $N$ (Nodes)', fontsize=14)
plt.ylabel('Log of Total Number of Links $L(N)$', fontsize=14)

# Log-Log scale (optional, but usually best for network edges!)
plt.yscale('log')
plt.xscale('log')

# Legend and grid
plt.legend(fontsize=11, frameon=True, shadow=True, loc='upper left')
plt.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()

plt.show()

Here we compute the theoretical degree distribution via the recurrence relation $p_k = \frac{(1+pk)}{(2+pk)} p_{k-1}$
We set a fixed number t, and compute the whole sequence of probability distribution followed by normalization.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from random import choice

# ==========================================
# 1. Core Model Class Definition
# ==========================================
class AllOrNoneDynamics:
    def __init__(self, p, initial_nodes=1):
        self.graph = nx.Graph()
        self.graph.add_nodes_from(range(initial_nodes))
        self.current_node_id = initial_nodes  
        self.threshold = p 
    
    def select_target_node(self):
        return choice(list(self.graph.nodes))
    
    def add_new_node(self):
        target_node = self.select_target_node()
        new_node = self.current_node_id
        self.graph.add_node(new_node)
        self.current_node_id += 1
        
        if np.random.uniform(0, 1) < self.threshold:
            # Clone
            connect_nodes = [target_node] + list(self.graph.neighbors(target_node))
        else:
            # Leaf
            connect_nodes = [target_node]
        
        connect_nodes = list(set(connect_nodes))  
        for node in connect_nodes:
            self.graph.add_edge(new_node, node)
            
    def run_dynamics(self, total_nodes=20):
        while len(self.graph.nodes) < total_nodes:
            self.add_new_node()

# ==========================================
# 2. Simulation Parameters
# ==========================================
p = 0.4  
N_max = 2000  
n_iter = 70  
np.random.seed(42)

# ==========================================
# 3. Run Multiple Simulations
# ==========================================
all_degree_counts = defaultdict(int)

print(f"Running {n_iter} simulations of {N_max} nodes...")
for i in range(n_iter):
    model = AllOrNoneDynamics(initial_nodes=1, p=p)
    model.run_dynamics(total_nodes=N_max)
    
    # Extract degrees
    degrees = [d for _, d in model.graph.degree()]
    for deg in degrees:
        all_degree_counts[deg] += 1

# Reconstruct raw degrees array for the histogram
raw_degrees = []
for deg, count in all_degree_counts.items():
    raw_degrees.extend([deg] * count)

# Compute average empirical probability
total_nodes_all_runs = n_iter * N_max
empirical_pk = {
    deg: count / total_nodes_all_runs 
    for deg, count in sorted(all_degree_counts.items())
}
max_deg = max(empirical_pk.keys())

# ==========================================
# 4. Compute Theoretical p_k
# ==========================================
def compute_theoretical_pk(p, max_deg):
    theoretical_pk = {}
    theoretical_pk[1] = (1 - p) / (2 + p * 1)
    for k in range(2, max_deg + 1):
        theoretical_pk[k] = ((1 + p * k) * theoretical_pk[k - 1]) / (2 + p * k)
    total = sum(theoretical_pk.values())
    return {k: v / total for k, v in theoretical_pk.items()}

theoretical_pk = compute_theoretical_pk(p, max_deg)

# ==========================================
# 5. Plotting (TWO DIAGRAMS SIDE-BY-SIDE)
# ==========================================
sns.set_style("whitegrid", {"grid.linestyle": "--", "grid.alpha": 0.5})
plt.rcParams.update({
    'font.sans-serif': ['Arial'],
    'axes.unicode_minus': False,
    'font.size': 11,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'medium'
})

# Create a figure with 1 row and 2 columns
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))
fig.suptitle(f'Degree Distribution Analysis (p={p}, N={N_max})', fontsize=16, y=1.05)

degs_sorted = sorted(empirical_pk.keys())
emp_probs = [empirical_pk[d] for d in degs_sorted]
theo_probs = [theoretical_pk[d] for d in degs_sorted]

# ------------------------------------------
# DIAGRAM 1: Scatter and Line Plot (PMF)
# ------------------------------------------
ax1.scatter(degs_sorted, emp_probs, label=f'Empirical PMF ({n_iter} runs)', 
            color='#1f77b4', alpha=0.8, s=40, edgecolors='black', linewidth=0.5)
ax1.plot(degs_sorted, theo_probs, label='Theoretical PMF', 
         color='#d62728', linestyle='--', linewidth=2)

ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('Degree $k$', fontsize=13)
ax1.set_ylabel('Probability $p(k)$', fontsize=13)
ax1.set_title('Probability Mass Function(log-log)', fontsize=14, pad=10)
ax1.legend(fontsize=11, frameon=True, shadow=True)
ax1.grid(True, which="both", ls="--", alpha=0.3)

# ------------------------------------------
# DIAGRAM 2: Histogram of Degree Density
# ------------------------------------------
# Create logarithmically spaced bins so the bars look nice on a log-log plot
min_deg, max_deg = min(degs_sorted), max(degs_sorted)
log_bins = np.logspace(np.log10(min_deg), np.log10(max_deg), num=40)

weights = np.ones_like(raw_degrees) / len(raw_degrees)
ax2.hist(raw_degrees, bins=log_bins, weights=weights, alpha=0.6, color='#7f7f7f', edgecolor='black', linewidth=0.8)

# Plot the theoretical line over it for reference (optional, but looks good)
ax2.plot(degs_sorted, theo_probs, color='#d62728', linestyle='-', linewidth=1.5, alpha=0.8, label='Theoretical Curve')

ax2.set_xlabel('Degree $k$', fontsize=13)
ax2.set_ylabel('Density', fontsize=13)
ax2.set_title('Degree Histogram(linear scale)', fontsize=14, pad=10)
ax2.legend(fontsize=11, frameon=True, shadow=True)
ax2.grid(True, which="both", ls="--", alpha=0.3)

# ------------------------------------------
# Final Layout Adjustments
# ------------------------------------------
plt.tight_layout()
plt.show()

Next we focus on the average number of triangles and from there we try to deduce the global clustering coefficient

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Fixed Class Definition
# ==========================================
class AllOrNoneDynamics:
    def __init__(self, p):
        self.graph = nx.Graph()
        # Start at N=2 with 1 edge (L=1, Delta=0)
        self.graph.add_edge(0, 1)
        self.current_node_id = 2  
        self.p = p 
        # Track history starting from N=2
        self.triangle_history = [0] 
    
    def add_new_node(self):
        target = np.random.randint(0, self.current_node_id)
        new_node = self.current_node_id
        self.graph.add_node(new_node)
        self.current_node_id += 1
        
        if np.random.uniform(0, 1) < self.p:
            # Clone: connect to target and all its neighbors
            connect_nodes = [target] + list(self.graph.neighbors(target))
        else:
            # Leaf: connect ONLY to target
            connect_nodes = [target]
        
        for node in set(connect_nodes):
            self.graph.add_edge(new_node, node)
            
        # Optimization: sum local triangles and divide by 3 for global count
        total_triangles = sum(nx.triangles(self.graph).values()) // 3
        self.triangle_history.append(total_triangles)
                
    def run_dynamics(self, total_nodes):
        while self.current_node_id < total_nodes:
            self.add_new_node()
        return np.array(self.triangle_history, dtype=float)

# ==========================================
# 2. Theoretical Iteration
# ==========================================
def compute_theoretical_triangles(p, N_max):
    delta_vals = np.zeros(N_max + 1)
    d_vals = np.zeros(N_max + 1)
    L_vals = np.zeros(N_max + 1)
    
    # Base cases at N=2
    L_vals[2] = 1.0
    d_vals[2] = 1.0 # d = 2*L/N = 2*1/2 = 1
    delta_vals[2] = 0.0
    
    for t in range(3, N_max + 1):
        prev_t = t - 1
        # Recurrence: Delta(t) = Delta(t-1) + p*d(t-1) + 3p*Delta(t-1)/(t-1)
        delta_vals[t] = delta_vals[prev_t] + p * d_vals[prev_t] + (3 * p * delta_vals[prev_t] / prev_t)
        
        # Update Links and Average Degree for the next step
        L_vals[t] = L_vals[prev_t] + 1 + p * d_vals[prev_t]
        d_vals[t] = 2 * L_vals[t] / t
        
    return delta_vals[2:] 

# ==========================================
# 3. Simulation & Plotting
# ==========================================
N_max = 1000
n_iter = 30
p_values = [0.1, 0.3, 0.5, 0.7] # Reduced set for faster plotting

sns.set_style("whitegrid", {"grid.linestyle": "--", "grid.alpha": 0.5})
plt.figure(figsize=(12, 8))
colors = sns.color_palette("husl", len(p_values))
N_range = np.arange(2, N_max + 1)

for idx, p in enumerate(p_values):
    print(f"Simulating p = {p}...")
    
    empirical_runs = []
    for _ in range(n_iter):
        model = AllOrNoneDynamics(p=p)
        # Returns history from N=2 to N_max
        history = model.run_dynamics(total_nodes=N_max)
        empirical_runs.append(history)
    
    # np.mean now works because history is float64, not Unicode strings
    empirical_avg = np.mean(empirical_runs, axis=0)
    theoretical_delta = compute_theoretical_triangles(p, N_max)
    
    plt.plot(N_range, empirical_avg, label=f'Empirical (p={p})', 
             color=colors[idx], linewidth=2.5, alpha=0.8)
    
    plt.plot(N_range, theoretical_delta, 
             color=colors[idx], linestyle='--', linewidth=2)

plt.plot([], [], color='gray', linewidth=2.5, label='Empirical (Averaged)')
plt.plot([], [], color='gray', linestyle='--', linewidth=2, label='Theoretical Equation')

plt.title(f'Total Number of Triangles $\Delta(N)$ vs. Network Size', fontsize=16, fontweight='bold')
plt.xlabel('Network Size $N$ (Nodes)', fontsize=14)
plt.ylabel('Total Triangles', fontsize=14)
plt.legend(fontsize=11, frameon=True, shadow=True, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import networkx as nx

# ==========================================
# 1. Fixed Class Definition
# ==========================================
class AllOrNoneDynamics:
    def __init__(self, p):
        self.graph = nx.Graph()
        self.graph.add_edge(0, 1)
        self.current_node_id = 2  
        self.p = p 
        self.triangle_history = [0] 
    
    def add_new_node(self):
        target = np.random.randint(0, self.current_node_id)
        new_node = self.current_node_id
        self.graph.add_node(new_node)
        self.current_node_id += 1
        
        if np.random.uniform(0, 1) < self.p:
            connect_nodes = [target] + list(self.graph.neighbors(target))
        else:
            connect_nodes = [target]
        
        for node in set(connect_nodes):
            self.graph.add_edge(new_node, node)
            
        total_triangles = sum(nx.triangles(self.graph).values()) // 3
        self.triangle_history.append(total_triangles)
                
    def run_dynamics(self, total_nodes):
        while self.current_node_id < total_nodes:
            self.add_new_node()
            # S_3 is the sum of cubed degrees
            S_3 = sum(d**3 for n, d in self.graph.degree())

# Omega is the sum of degree products across edges
            Omega = sum(self.graph.degree(u) * self.graph.degree(v) for u, v in self.graph.edges())
        return np.array(self.triangle_history, dtype=float)

# ==========================================
# 2. Theoretical Iteration
# ==========================================
def compute_theoretical_triangles(p, N_max):
    delta_vals = np.zeros(N_max + 1)
    d_vals = np.zeros(N_max + 1)
    L_vals = np.zeros(N_max + 1)
    
    L_vals[2] = 1.0
    d_vals[2] = 1.0 
    delta_vals[2] = 0.0
    
    for t in range(3, N_max + 1):
        prev_t = t - 1
        delta_vals[t] = delta_vals[prev_t] + p * d_vals[prev_t] + (3 * p * delta_vals[prev_t] / prev_t)
        
        L_vals[t] = L_vals[prev_t] + 1 + p * d_vals[prev_t]
        d_vals[t] = 2 * L_vals[t] / t
        
    return delta_vals[2:] 

# ==========================================
# 3. Execute and Save
# ==========================================
N_max = 1000
n_iter = 30
p_values = [0.1, 0.3, 0.5, 0.7]

N_range = np.arange(2, N_max + 1)

# Dictionary to hold all the arrays we want to save
data_dict = {'N_range': N_range, 'p_values': p_values}

for p in p_values:
    print(f"Simulating p = {p}...")
    empirical_runs = []
    for _ in range(n_iter):
        model = AllOrNoneDynamics(p=p)
        history = model.run_dynamics(total_nodes=N_max)
        empirical_runs.append(history)
    
    empirical_avg = np.mean(empirical_runs, axis=0)
    theoretical_delta = compute_theoretical_triangles(p, N_max)
    
    # Store results dynamically using the p-value in the key name
    data_dict[f'emp_{p}'] = empirical_avg
    data_dict[f'theo_{p}'] = theoretical_delta

# Save everything into a single compressed NPZ file
np.savez('triangle_data.npz', **data_dict)
print("Finished! Data successfully saved to 'triangle_data.npz'")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Load the pre-computed data
# ==========================================
data = np.load('triangle_data.npz')
N_range = data['N_range']
p_values = data['p_values']

# ==========================================
# 2. Plotting (Tweak this freely!)
# ==========================================
sns.set_style("whitegrid", {"grid.linestyle": "--", "grid.alpha": 0.5})
plt.figure(figsize=(12, 8))
colors = sns.color_palette("husl", len(p_values))

for idx, p in enumerate(p_values):
    # Extract the specific arrays for this p-value
    empirical_avg = data[f'emp_{p}']
    theoretical_delta = data[f'theo_{p}']
    
    plt.plot(N_range, empirical_avg, label=f'Empirical (p={p})', 
             color=colors[idx], linewidth=2.5, alpha=0.8)
    
    plt.plot(N_range, theoretical_delta, 
             color=colors[idx], linestyle='--', linewidth=2)

# Legend formatting tricks
plt.plot([], [], color='gray', linewidth=2.5, label='Empirical (Averaged)')
plt.plot([], [], color='gray', linestyle='--', linewidth=2, label='Theoretical Equation')

# Axis formatting
plt.title('Total Number of Triangles $\\Delta(N)$ vs. Network Size', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Network Size $N$ (Nodes)', fontsize=14)
plt.ylabel('Total Triangles', fontsize=14)

# Log scales are highly recommended for N^{3p} growth
plt.yscale('log')
plt.xscale('log')

plt.legend(fontsize=11, frameon=True, shadow=True, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# For Clustering Coefficient
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Core Model Class Definition
# ==========================================
class AllOrNoneDynamics:
    """
    All-or-None Dynamics Model with Interval Clustering Tracking
    """
    def __init__(self, p):
        self.graph = nx.Graph()
        # Initialize with 2 nodes and 1 edge
        self.graph.add_edge(0, 1)
        self.current_node_id = 2  
        self.p = p 
    
    def add_new_node(self):
        target = np.random.randint(0, self.current_node_id)
        new_node = self.current_node_id
        self.graph.add_node(new_node)
        self.current_node_id += 1
        
        judge_value = np.random.uniform(0, 1)
        
        if judge_value < self.p:
            # Clone (Probability p)
            connect_nodes = [target] + list(self.graph.neighbors(target))
        else:
            # Leaf (Probability 1-p)
            connect_nodes = [target]
        
        # Establish connections
        connect_nodes = list(set(connect_nodes))  
        for node in connect_nodes:
            self.graph.add_edge(new_node, node)
            
    def run_dynamics(self, total_nodes, sample_points):
        """
        Runs the growth and calculates the clustering coefficient ONLY 
        at the specified sample points to save computation time.
        """
        history = []
        while self.current_node_id <= total_nodes:
            if self.current_node_id in sample_points:
                # Transitivity = 3 * Triangles / Triplets (Global Clustering)
                c = nx.transitivity(self.graph)
                history.append(c)
                
            if self.current_node_id < total_nodes:
                self.add_new_node()
            else:
                break
                
        return np.array(history)

# ==========================================
# 2. Simulation & Plotting
# ==========================================
N_max = 1500  # Go slightly higher to observe the plateau
n_iter = 20   # 20 runs to smooth the curve
p_values = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

# Sample every 50 nodes to keep the simulation fast
sample_points = np.arange(50, N_max + 1, 50)

sns.set_style("whitegrid", {"grid.linestyle": "--", "grid.alpha": 0.5})
plt.figure(figsize=(12, 8))
colors = sns.color_palette("Set1", len(p_values))

for idx, p in enumerate(p_values):
    print(f"Simulating p = {p}...")
    
    # 1. Gather Empirical Data
    empirical_runs_clustering = []
    for _ in range(n_iter):
        model = AllOrNoneDynamics(p=p)
        history = model.run_dynamics(total_nodes=N_max, sample_points=sample_points)
        empirical_runs_clustering.append(history)
    
    # Average across all iterations
    empirical_avg_c = np.mean(empirical_runs_clustering, axis=0)
    
    # 2. Theoretical Asymptote (User's Deduction)
    theoretical_c = (3 * p) / (1 + 2 * p)
    
    # 3. Plotting
    # Empirical (Solid line)
    plt.plot(sample_points, empirical_avg_c, label=f'Empirical (p={p})', 
             color=colors[idx], linewidth=2.5, alpha=0.8)
    
    # Theoretical Asymptote (Horizontal Dashed Line)
    plt.axhline(y=theoretical_c, color=colors[idx], linestyle='--', 
                linewidth=2, dash_capstyle='round', dashes=[4, 4])

# Fake lines for legend formatting
plt.plot([], [], color='gray', linewidth=2.5, label='Empirical Convergence')
plt.plot([], [], color='gray', linestyle='--', linewidth=2, label='Theoretical $C = 3p / (1+2p)$')

# Formatting
plt.title(f'Global Clustering Coefficient vs. Network Size\n(Averaged over {n_iter} runs)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Network Size $N$ (Nodes)', fontsize=14)
plt.ylabel('Clustering Coefficient (Transitivity)', fontsize=14)

# Linear scales are best here since C is bounded between 0 and 1
plt.ylim(0, 1)
plt.xlim(0, N_max)

# Legend and grid
plt.legend(fontsize=11, frameon=True, shadow=True, loc='lower left')
plt.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Stochastic Simulation (Empirical)
# ==========================================
class AllOrNoneDynamics:
    def __init__(self, p):
        self.graph = nx.Graph()
        self.graph.add_edge(0, 1)
        self.current_node_id = 2  
        self.p = p 
    
    def add_new_node(self):
        target = np.random.randint(0, self.current_node_id)
        new_node = self.current_node_id
        self.graph.add_node(new_node)
        self.current_node_id += 1
        
        if np.random.uniform(0, 1) < self.p:
            connect_nodes = [target] + list(self.graph.neighbors(target))
        else:
            connect_nodes = [target]
        
        for node in set(connect_nodes):
            self.graph.add_edge(new_node, node)

def run_multiple_empirical(p, N_max, step=100, iterations=20):
    N_checkpoints = range(500, N_max + 1, step)
    results_at_N = {n: [] for n in N_checkpoints}
    
    for i in range(iterations):
        print(f"    -> Running iteration {i+1}/{iterations} for p={p}...", end='\r')
        model = AllOrNoneDynamics(p)
        while model.current_node_id <= N_max:
            model.add_new_node()
            N = model.current_node_id
            if N in results_at_N:
                try:
                    r = nx.degree_assortativity_coefficient(model.graph)
                    results_at_N[N].append(r)
                except:
                    pass
    print() 
    N_vals = sorted(list(results_at_N.keys()))
    r_means = [np.mean(results_at_N[n]) if results_at_N[n] else 0 for n in N_vals]
    return N_vals, np.array(r_means)

# ==========================================
# 2. Master Equations (Theoretical ODE Integration)
# ==========================================
def run_theoretical(p, N_max, step=100):
    L, T, S2, S3, Omega = 1.0, 0.0, 2.0, 2.0, 1.0
    N_vals, r_vals = [], []
    for n in range(2, N_max + 1):
        dL = 1 + (2 * p / n) * L
        dT = (p * 2 * L / n) + (3 * p / n) * T
        dS2 = (3 * p / n) * S2 + ((4 + 6 * p) / n) * L + 2
        dS3 = (4 * p / n) * S3 + ((3 + 6 * p) / n) * S2 + ((6 + 8 * p) / n) * L + 2
        dOmega = (4 * p / n) * Omega + ((1 + 3 * p) / n) * (S2 + 2 * L) + 1 + (3 * p / n) * T
        L += dL; T += dT; S2 += dS2; S3 += dS3; Omega += dOmega
        if n % step == 0 and n >= 500:
            num = S3 - 2 * Omega
            den = S3 - (S2**2) / (2 * L)
            r = 1 - (num / den) if den > 1e-10 else 0
            N_vals.append(n)
            r_vals.append(r)
    return N_vals, r_vals, r_vals[-1]

# ==========================================
# 3. Execution & Plotting
# ==========================================
N_max_theo = 10000   
N_max_emp = 10000     
iterations = 10      # Reduced iterations slightly for faster rendering demo

p_values = [0.10, 0.20, 0.40, 0.60] # Subset for cleaner legend, add more if needed
colors = ['navy', 'deepskyblue', 'goldenrod', 'firebrick']
color_map = dict(zip(p_values, colors))

sns.set_style("whitegrid")
plt.figure(figsize=(12, 7))

for p in p_values:
    c = color_map[p]
    
    # 1. Theoretical Data
    N_theo, r_theo, final_theo_r = run_theoretical(p, N_max_theo, step=100)
    plt.plot(N_theo, r_theo, color=c, linewidth=2.5, solid_capstyle='round',
             label=f'Theory p={p} (Limit {final_theo_r:.2f})')
    
    # 2. Empirical Data
    N_emp, r_mean = run_multiple_empirical(p, N_max_emp, step=500, iterations=iterations)
    plt.plot(N_emp, r_mean, color=c, linewidth=1, linestyle='--', marker='o', 
             markersize=4, alpha=0.8, label=f'Empirical p={p}')

# ==========================================
# 4. Final Formatting
# ==========================================
plt.title('Assortativity $r$: Theory vs Empirical Validation', fontsize=16, fontweight='bold')
plt.xlabel('Network Size $N$', fontsize=12)
plt.ylabel('Assortativity Coefficient $r$', fontsize=12)
plt.ylim(0, 1.05)
plt.xlim(500, N_max_theo)

# Sophisticated legend placement
plt.legend(title="Dynamics Models", title_fontsize='13', fontsize=10, 
           loc='upper left', bbox_to_anchor=(1.1, 1), frameon=True, shadow=True)

# Secondary Axis for Asymptotes
ax2 = plt.gca().twinx()
ax2.set_ylim(0, 1.05)
asymptotes = [run_theoretical(p, N_max_theo, 100)[2] for p in p_values]
ax2.set_yticks(asymptotes)
ax2.set_yticklabels([f"{val:.2f}" for val in asymptotes], fontsize=9, color='gray')
ax2.set_ylabel('Theoretical Limits', color='gray', rotation=270, labelpad=15)
ax2.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_all_or_none(N_max, p, num_trials=50):
    """
    Simulates the all-or-none network model and returns the average 
    shortest path length L(N) from N=2 to N_max.
    """
    L_avg = np.zeros(N_max + 1)
    
    for trial in range(num_trials):
        D = np.zeros((N_max + 1, N_max + 1), dtype=np.int32)
        
        # Exact initial condition: N=2 nodes connected by 1 edge
        D[1, 2] = 1
        D[2, 1] = 1
        
        current_distance_sum = 1 
        L_trial = np.zeros(N_max + 1)
        L_trial[2] = 1.0 
        
        for n in range(3, N_max + 1):
            t = np.random.randint(1, n)
            is_clone = np.random.rand() < p
            
            if is_clone:
                new_distances = D[t, 1:n].copy()
                new_distances[t-1] = 1 
            else:
                new_distances = D[t, 1:n] + 1
                new_distances[t-1] = 1 
                
            D[n, 1:n] = new_distances
            D[1:n, n] = new_distances
            
            current_distance_sum += np.sum(new_distances)
            num_pairs = (n * (n - 1)) / 2
            L_trial[n] = current_distance_sum / num_pairs
            
        L_avg += L_trial
        
    return L_avg / num_trials

# ==========================================
# Run the simulation and plot against theory
# ==========================================
N_max = 1500
p_values = [0.2,0.3, 0.4,0.5, 0.6, 0.7, 0.8]
num_trials = 100 

# Precompute Harmonic numbers for the exact theory
H = np.zeros(N_max + 1)
for i in range(1, N_max + 1):
    H[i] = H[i-1] + 1/i

# Setup the figure
plt.figure(figsize=(12, 8))

for p in p_values:
    # 1. Run robust simulation
    L_sim = simulate_all_or_none(N_max, p, num_trials)
    
    # 2. Calculate Exact Theory
    N = np.arange(2, N_max + 1)
    L_exact = ((N + 1) / (3 * (N - 1))) * (6 * (1 - p) * H[N] + 15 * p - 12) + (2 * (2 - 3 * p) / (N - 1))
    
    # Plot Simulation (solid line) and extract the color Matplotlib chooses
    sim_line = plt.plot(N, L_sim[2:], label=f'Simulation (p={p})', alpha=0.8, linewidth=2)[0]
    matched_color = sim_line.get_color()
    
    # Plot Exact Theory (dashed line) using the exact same color
    plt.plot(N, L_exact, '--', label=f'Exact Theory (p={p})', color=matched_color, linewidth=2.5)

# ==========================================
# Formatting: Titles, Labels, Legend, & Grid
# ==========================================
plt.title('Average Shortest Path in All-or-None Networks\nSimulation vs Exact Theory', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Graph Size $N$ (number of nodes)', fontsize=14)
plt.ylabel('Average Shortest Path Length $L(N)$', fontsize=14)

# Customize the legend to make it easy to read
plt.legend(loc='upper left', fontsize=12, framealpha=0.9, edgecolor='gray')

# Add a subtle grid
plt.grid(True, linestyle=':', alpha=0.7)

# Adjust layout to prevent clipping
plt.tight_layout()

# Display the plot
plt.show()